[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_09_Hyperparameter_Tuning/02_pipelines.ipynb)

# Episode 23 – ML Pipelines: Putting It All Together

**Machine Learning Bootcamp** | Module 09

---

## 🎯 Learning Objectives
- Build end-to-end reproducible ML pipelines with scikit-learn
- Combine preprocessing and modelling steps safely
- Save and reload a trained pipeline for inference

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report

sns.set_theme(style='whitegrid')

## 1. Why Pipelines?

Without a pipeline:
```python
# ❌ Data leakage risk – scaler fitted on ALL data
scaler.fit(X)                 # sees test data!
X_scaled = scaler.transform(X)
model.fit(X_scaled, y)
```

With a pipeline:
```python
# ✅ Safe – scaler fitted only on training fold
pipe = Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier())])
cross_val_score(pipe, X, y, cv=5)
```

## 2. Create a Mixed-Type Dataset

In [ ]:
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'age':       np.random.randint(18, 70, n).astype(float),
    'income':    np.random.normal(50000, 15000, n),
    'city':      np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n),
    'gender':    np.random.choice(['M', 'F'], n),
    'purchased': np.random.randint(0, 2, n)
})

# Introduce missing values
df.loc[np.random.choice(n, 30, replace=False), 'age']    = np.nan
df.loc[np.random.choice(n, 20, replace=False), 'income'] = np.nan

X = df.drop('purchased', axis=1)
y = df['purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training samples:', X_train.shape[0], '| Test samples:', X_test.shape[0])
X.head()

## 3. Build the Pipeline with ColumnTransformer

In [ ]:
numerical_features   = ['age', 'income']
categorical_features = ['city', 'gender']

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer,   numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

# Full pipeline
pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

pipe

## 4. Train & Evaluate

In [ ]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print()
print(classification_report(y_test, y_pred))

## 5. Save & Reload the Pipeline

In [ ]:
# Save
joblib.dump(pipe, 'trained_pipeline.pkl')
print('Pipeline saved to trained_pipeline.pkl')

# Reload and predict
loaded_pipe = joblib.load('trained_pipeline.pkl')
sample = pd.DataFrame([{'age': 35, 'income': 60000, 'city': 'NYC', 'gender': 'F'}])
print('Sample prediction:', loaded_pipe.predict(sample)[0])
print('Sample probability:', loaded_pipe.predict_proba(sample)[0].round(4))

## 🏋️ Exercises

1. Add a `GridSearchCV` on top of the pipeline to tune `classifier__n_estimators` and `classifier__max_depth`.
2. Replace `RandomForestClassifier` with `LogisticRegression` in the pipeline. Does the CV accuracy change?
3. Add a `FunctionTransformer` step to the numerical pipeline that clips income to [0, 100000] before scaling.

---

## 🎉 Congratulations!

You have completed the **Machine Learning Bootcamp**!

### What You've Learned
- ✅ Foundations of ML and Python tools
- ✅ Data preprocessing and feature engineering
- ✅ Supervised learning (regression & classification)
- ✅ Model evaluation and validation
- ✅ Unsupervised learning (clustering & dimensionality reduction)
- ✅ Neural networks and deep learning with Keras
- ✅ Hyperparameter tuning and reproducible ML pipelines

### Next Steps
- 🔬 Apply these skills to a Kaggle competition
- 🧠 Explore CNNs, RNNs, and Transformers
- 🚀 Learn MLOps and model deployment